# SmolVLM MMR at 32 visual tokens after decoder block 23

Choose **GPU T4 ×2**, enable Internet, and use **Save & Run All**. The notebook downloads the pinned model and data, then builds shared visual features without a Kaggle Input.

All configurations process the 384 visual tokens through block 23 (`layers[22]`). After block 23, they retain 32 visual tokens for block 24 onward. Raw top-32 uses attention from the final prompt token at block 23. The three MMR methods apply per-question min–max scaling to that same relevance and compute redundancy from the visual hidden states output by block 23.

- **Raw MMR** uses hidden-state cosine across all visual tokens.
- **Intra-camera MMR** applies the penalty only within the same camera.
- **Spatial-aware MMR** weights hidden-state cosine by distance on the same camera's 8×8 grid. Cross-camera pairs keep half their cosine.

Each MMR method runs at λ = 0.9. The four configurations use the same 4,204 stratified questions and FP32 greedy decoding. Positional bias calibration is skipped. The report shows quality and latency deltas against the raw top-32 reference run at block 23, with a bar chart of Overall score deltas by method.

In [ ]:
GPU_COUNT = 2
GENERATION_BATCH = 4  # Per GPU, with automatic CUDA OOM backoff.
KEEP_COUNTS = (32,)
EVALUATION_SUBSET_SIZE = 4204  # Set to None to use the full 8,408-question cohort.
MMR_ATTENTION_NORMALIZATION = 'minmax'  # 'minmax' or 'none'; applies only to MMR relevance.
MMR_LAMBDA = 0.9
PRUNING_LAYER = 23  # One-based decoder block number, index 22.
MMR_METHODS = ('raw_mmr', 'intra_camera_mmr', 'spatial_aware_mmr')
SELECTION_EXAMPLES = 3

In [ ]:
import json
import os
from base64 import b64decode
from io import BytesIO
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
if shutil.which('java') is None:
    raise RuntimeError('Java is needed for COCO caption evaluation.')
SOURCE = Path(tempfile.mkdtemp(prefix='log-metrics-raw-mmr-'))
SOURCE_B64 = '...'
with zipfile.ZipFile(BytesIO(b64decode(SOURCE_B64))) as archive:
    archive.extractall(SOURCE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt'), 'pycocoevalcap==1.2'], check=True)
sys.path.insert(0, str(SOURCE))
protocol = json.loads((SOURCE / 'raw_mmr_protocol.json').read_bytes())

In [ ]:
import tokenizers
import torch
import transformers
from huggingface_hub import hf_hub_download
from smolvlm_attention.config import MODELS, PREPARED_REPO, PREPARED_REVISION
from smolvlm_attention.posprune_ablation.image_features import IMAGE_SOURCE
from smolvlm_attention.posprune_ablation.runtime import GENERATION, SHALLOW
from smolvlm_inference.cache import digest

if (transformers.__version__, tokenizers.__version__) != ('4.46.3', '0.20.4'):
    raise RuntimeError('Restart the kernel after dependency installation and Run All.')
worker_count = min(GPU_COUNT, torch.cuda.device_count())
if worker_count < 1:
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
base = MODELS['base']
if (GENERATION != protocol['generation'] or SHALLOW != protocol['shallow']
        or base['camera_order'] != protocol['camera_order']
        or any(protocol['feature_source'].get(key) != value for key, value in IMAGE_SOURCE.items())
        or protocol['feature_source']['mode'] != 'regenerated'):
    raise RuntimeError('The notebook configuration differs from the historical experiment.')

work = Path('/kaggle/working/smolvlm-mmr-sweep-layer23-32')
prepared = work / 'prepared_data'
for name in ('manifest.json', 'test.jsonl'):
    hf_hub_download(repo_id=PREPARED_REPO, repo_type='dataset', revision=PREPARED_REVISION,
                    filename=name, local_dir=prepared, token=False)
checkpoint = Path(hf_hub_download(repo_id=base['repo'], revision=base['revision'],
                                 filename='finetune_best.pt', token=False))
if digest(checkpoint) != protocol['checkpoint_sha256']:
    raise RuntimeError('The checkpoint SHA-256 differs from the historical experiment.')

print(f'Using {worker_count} GPU(s). Budgets {KEEP_COUNTS}, MMR normalization {MMR_ATTENTION_NORMALIZATION}.')

In [ ]:
from smolvlm_attention.posprune_ablation.parallel import run_parallel

run_dir = Path(tempfile.mkdtemp(prefix='run-', dir=work))
output = run_parallel(checkpoint, None, prepared, run_dir / 'results', cache_fingerprint=None,
    camera_order=base['camera_order'], gpu_count=worker_count, evaluation_size=None,
    evaluation_fraction=protocol['evaluation_fraction'], seed=protocol['seed'],
    evaluation_subset_size=EVALUATION_SUBSET_SIZE,
    generation_batch=GENERATION_BATCH, keep_counts=KEEP_COUNTS, selection_examples=SELECTION_EXAMPLES,
    raw_mmr=True, expected_evaluation_ids=protocol['evaluation_ids'],
    attention_normalization=MMR_ATTENTION_NORMALIZATION, mmr_sweep=(MMR_LAMBDA,),
    pruning_layer=PRUNING_LAYER, mmr_sweep_methods=MMR_METHODS)
print((output / 'parallel_timing.json').read_text())

In [ ]:
import csv
from IPython.display import FileLink, Image, display
from smolvlm_attention.posprune_ablation.raw_mmr_report import report_raw_mmr

for path in report_raw_mmr(output):
    if path.suffix == '.png':
        display(Image(filename=str(path)))
with (output / 'mmr_vs_raw.csv').open() as stream:
    for row in csv.DictReader(stream):
        print(f"Keep {row['kept_tokens']} · {row['method']} · λ={row['mmr_lambda']}: Overall {float(row['raw_overall']):.3f} → "
              f"{float(row['mmr_overall']):.3f} (Δ {float(row['delta_overall']):+.3f})")
archive = run_dir / 'raw-mmr-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.is_file() and path.name != 'generation_ready.json':
            bundle.write(path, path.name)
for path in (output / 'comparison.csv', output / 'mmr_vs_raw.csv', output / 'paired_answers.csv', archive):
    display(FileLink(os.path.relpath(path)))